# Hawaii Mesonet — Threshold Violation Chart
Run the cells in order. Only edit the **SETTINGS** cell.

In [ ]:
# ============================================================
# SETTINGS — edit these, then run all cells
# ============================================================

STATION_ID   = '9001'         # Station ID to plot
VARIABLE     = 'rh_1'         # Variable to plot (see list below)

# Available variables:
#   air_temp_1, air_temp_2, rh_1, rh_2
#   sw_in, sw_out, lw_in, lw_out
#   wind_speed, wind_dir, air_pressure, surface_temp
#   soil_vwc_1, soil_vwc_2, soil_vwc_3
#   soil_heat_flux, soil_hf_2
#   tsoil_1, tsoil_2, tsoil_3, tsoil_4
#   rainfall_1, rainfall_2

# ============================================================
# PATHS
# ============================================================

STATION_DATA_FOLDER = '../data/stationdata'
JSON_REPORT         = 'output/threshold_report.json'


In [ ]:
# ── Imports ──────────────────────────────────────────────────
import pandas as pd
import json
import os
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print('Libraries loaded.')


In [ ]:
# ── Variable group definitions ────────────────────────────────
# When you pick a variable, all related variables are plotted together.

CHART_GROUPS = [
    ['air_temp_1', 'air_temp_2', 'rh_1', 'rh_2'],
    ['sw_in', 'sw_out'],
    ['lw_in', 'lw_out'],
    ['soil_vwc_1', 'soil_vwc_2', 'soil_vwc_3', 'rainfall_1', 'rainfall_2'],
    ['tsoil_1', 'tsoil_2', 'tsoil_3', 'tsoil_4'],
    ['soil_heat_flux', 'soil_hf_2'],
    ['wind_speed'],
    ['wind_dir'],
    ['air_pressure'],
    ['surface_temp'],
]

# Column name aliases — maps norm_name to possible CSV column headers
COL_ALIASES = {
    'sw_in':          ['Shortwave incoming radiation (W/m²)', 'Incoming shortwave radiation (W/m²)', 'SWin_1_Avg'],
    'sw_out':         ['Shortwave outgoing radiation (W/m²)', 'Outgoing shortwave radiation (W/m²)', 'SWout_1_Avg'],
    'lw_in':          ['Longwave incoming radiation (W/m²)', 'Incoming longwave radiation (W/m²)', 'LWin_1_Avg'],
    'lw_out':         ['Longwave outgoing radiation (W/m²)', 'Outgoing longwave radiation (W/m²)', 'LWout_1_Avg'],
    'air_temp_1':     ['Air temperature, sensor 1 (°C)', 'Tair_1_Avg'],
    'air_temp_2':     ['Air temperature, sensor 2 (°C)', 'Tair_2_Avg'],
    'rh_1':           ['Relative humidity, sensor 1 (%)', 'RH_1_Avg'],
    'rh_2':           ['Relative humidity, sensor 2 (%)', 'RH_2_Avg'],
    'wind_speed':     ['Wind speed, scalar average (m/s)', 'WS_1_Avg'],
    'wind_dir':       ['Unit vector wind direction (°)', 'Wind direction, scalar average (°)', 'WDuv_1_Avg'],
    'air_pressure':   ['Air pressure (kPa)', 'P_1_Avg'],
    'surface_temp':   ['Surface temperature (°C)', 'Tsrf_1_Avg'],
    'soil_vwc_1':     ['Soil volumetric water content (m³/m³)', 'SM_1_Avg'],
    'soil_vwc_2':     ['Soil volumetric water content, sensor 2 (m³/m³)', 'SM_2_Avg'],
    'soil_vwc_3':     ['Soil volumetric water content, sensor 3 (m³/m³)', 'SM_3_Avg'],
    'soil_heat_flux': ['Soil heat flux at ground surface (W/m²)', 'SHFsrf_1_Avg'],
    'soil_hf_2':      ['Soil heat flux, depth 2 (W/m²)', 'Soil heat flux below ground, sensor 2 (W/m²)', 'SHF_2_Avg'],
    'tsoil_1':        ['Soil temperature, depth 1 (°C)', 'Soil temperature (°C)', 'Tsoil_1_Avg'],
    'tsoil_2':        ['Soil temperature, depth 2 (°C)', 'Soil temperature, sensor 2 (°C)', 'Tsoil_2_Avg'],
    'tsoil_3':        ['Soil temperature, depth 3 (°C)', 'Soil temperature, sensor 3 (°C)', 'Tsoil_3_Avg'],
    'tsoil_4':        ['Soil temperature, depth 4 (°C)', 'Soil temperature, sensor 4 (°C)', 'Tsoil_4_Avg'],
    'rainfall_1':     ['Rainfall, 5-minute total (mm)', 'RF_1_Tot300s'],
    'rainfall_2':     ['Rainfall total, sensor 2 (mm)', 'Rainfall, 5-minute total, sensor 2 (mm)', 'RF_2_Tot300s'],
}

# Valid ranges — synced with Threshold_Errors.py
THRESHOLDS = {
    'air_temp_1':     (-10,  40),
    'air_temp_2':     (-10,  40),
    'rh_1':           (0,    100),
    'rh_2':           (0,    100),
    'sw_in':          (0,    1400),
    'sw_out':         (0,    400),
    'lw_in':          (200,  500),
    'lw_out':         (200,  700),
    'wind_speed':     (0,    80),
    'wind_dir':       (0,    360),
    'air_pressure':   (60,   104),
    'surface_temp':   (0,    65),
    'rainfall_1':     (0,    150),
    'rainfall_2':     (0,    150),
    'soil_vwc_1':     (0,    0.9),
    'soil_vwc_2':     (0,    0.9),
    'soil_vwc_3':     (0,    0.9),
    'soil_heat_flux': (-400, 400),
    'soil_hf_2':      (-150, 150),
    'tsoil_1':        (5,    45),
    'tsoil_2':        (5,    45),
    'tsoil_3':        (5,    45),
    'tsoil_4':        (5,    45),
}

TRACE_COLORS = ['#e07b00', '#1a5c2e', '#2b6cb0', '#c53030', '#6b46c1', '#276749']

print('Configuration loaded.')


In [ ]:
# ── Load station CSV ──────────────────────────────────────────

csv_path = Path(STATION_DATA_FOLDER) / f'{STATION_ID}.csv'
if not csv_path.exists():
    raise FileNotFoundError(f'Station file not found: {csv_path}')

df = pd.read_csv(csv_path, low_memory=False)

# Find and parse timestamp column
ts_col = next((c for c in df.columns if 'timestamp' in c.lower()), None)
if ts_col is None:
    raise ValueError('No timestamp column found in CSV.')
df[ts_col] = pd.to_datetime(df[ts_col], errors='coerce')
df = df.sort_values(ts_col).reset_index(drop=True)

print(f'Loaded {len(df):,} rows from {csv_path.name}')
print(f'Date range: {df[ts_col].min()} → {df[ts_col].max()}')


In [ ]:
# ── Resolve variable group and column names ───────────────────

def get_group(variable):
    for g in CHART_GROUPS:
        if variable in g:
            return g
    return [variable]

def resolve_col(df_cols, norm_name):
    for alias in COL_ALIASES.get(norm_name, []):
        if alias in df_cols:
            return alias
    return None

group_vars = get_group(VARIABLE)
col_map    = {v: resolve_col(df.columns, v) for v in group_vars}
found      = {v: c for v, c in col_map.items() if c is not None}
missing    = [v for v, c in col_map.items() if c is None]

print(f'Variable group: {group_vars}')
print(f'Found in CSV:   {list(found.keys())}')
if missing:
    print(f'Not in CSV:     {missing} (will be skipped)')


In [ ]:
# ── Load flags from threshold_report.json ────────────────────
# Flags come from Threshold_Errors.py which uses correct per-station
# sunrise/sunset gating — no need to recompute here.

# Build reverse alias map: column_name → norm_name
col_to_norm = {}
for norm, aliases in COL_ALIASES.items():
    for alias in aliases:
        col_to_norm[alias] = norm

# Load JSON and filter to this station
report_path = Path(JSON_REPORT)
if not report_path.exists():
    raise FileNotFoundError(
        f'JSON report not found: {JSON_REPORT}\n'
        f'Run Threshold_Errors.py first to generate it.'
    )

with open(report_path) as f:
    report = json.load(f)

station_events = [e for e in report['events'] if e['station_id'] == STATION_ID]
print(f'Report generated: {report["generated"]}')
print(f'Events for station {STATION_ID}: {len(station_events)}')

# Build violation_map: norm_name → list of (start, end) pd.Timestamp spans
# Use sub_periods for exact bad-reading windows where available
violation_map = {}
for e in station_events:
    norm = col_to_norm.get(e['variable'])
    if norm is None:
        continue
    periods = e.get('sub_periods') or []
    if periods:
        spans = [(pd.Timestamp(p['start_time']), pd.Timestamp(p['end_time'])) for p in periods]
    else:
        spans = [(pd.Timestamp(e['start_time']), pd.Timestamp(e['end_time']))]
    violation_map.setdefault(norm, []).extend(spans)

for var in found:
    n_spans = len(violation_map.get(var, []))
    print(f'  {var}: {n_spans} violation span(s) from report')


In [ ]:
# ── Compute sunrise/sunset for day/night shading ─────────────
import ephem, math, csv
from datetime import timedelta

def _station_coords(station_id, csv_path):
    with open(csv_path, newline="", encoding="utf-8") as f:
        for row in __import__("csv").DictReader(f):
            if row["station_id"] == station_id:
                try:
                    return float(row["lat"]), float(row["lng"]),                            float(row["elevation"]), row.get("location","hawaii")
                except (ValueError, KeyError):
                    return None
    return None

def compute_night_periods(ts_series, station_id, metadata_path):
    """Return list of (sunset, sunrise) pd.Timestamps for each night in the data range."""
    coords = _station_coords(station_id, metadata_path)
    if coords is None:
        print(f"  WARNING: {station_id} not found in metadata — skipping day/night shading")
        return []
    lat, lon, elev, location = coords
    tz_h    = -11 if "samoa" in location.lower() else -10
    tz      = timedelta(hours=tz_h)
    noon_utc_h = 12 - tz_h

    _R = 6371000.0
    obs = ephem.Observer()
    obs.lat, obs.lon, obs.elevation = str(lat), str(lon), elev
    obs.horizon = str(math.degrees(-math.acos(_R / (_R + elev)))) if elev > 0 else "0"

    d_start = ts_series.min().date()
    d_end   = ts_series.max().date()

    nights, d = [], d_start
    prev_ss = None
    while d <= d_end + __import__("datetime").timedelta(days=1):
        obs.date = f"{d.year}/{d.month}/{d.day} {noon_utc_h:02d}:00:00"
        try:
            sr = pd.Timestamp(obs.previous_rising(ephem.Sun()).datetime()) + tz
            ss = pd.Timestamp(obs.next_setting(ephem.Sun()).datetime()) + tz
        except Exception:
            d += __import__("datetime").timedelta(days=1)
            continue
        if prev_ss is not None:
            nights.append((prev_ss, sr))   # night = prev sunset → this sunrise
        prev_ss = ss
        d += __import__("datetime").timedelta(days=1)

    return nights

METADATA_PATH = '../data/station_metadata.csv'
night_periods = compute_night_periods(df[ts_col], STATION_ID, METADATA_PATH)
print(f"Computed {len(night_periods)} night periods for station {STATION_ID}")


In [ ]:
# ── Build and display interactive chart ──────────────────────
import webbrowser, tempfile

plot_vars = list(found.keys())
n         = len(plot_vars)

if n == 0:
    print('ERROR: No variables found in CSV for this group. Check COL_ALIASES.')
else:
    fig = make_subplots(
        rows=n, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=plot_vars,
    )

    for row_i, var in enumerate(plot_vars, 1):
        col   = found[var]
        color = TRACE_COLORS[(row_i - 1) % len(TRACE_COLORS)]
        lo, hi = THRESHOLDS.get(var, (None, None))

        fig.add_trace(
            go.Scatter(
                x=df[ts_col],
                y=pd.to_numeric(df[col], errors='coerce'),
                name=var,
                line=dict(color=color, width=1.0),
                connectgaps=False,
            ),
            row=row_i, col=1,
        )

        if lo is not None:
            fig.add_hline(y=lo, line_dash='dash', line_color='#c53030', line_width=1.2,
                          annotation_text=f'Min {lo}', annotation_font_size=9,
                          row=row_i, col=1)
        if hi is not None:
            fig.add_hline(y=hi, line_dash='dash', line_color='#c53030', line_width=1.2,
                          annotation_text=f'Max {hi}', annotation_font_size=9,
                          row=row_i, col=1)

        MIN_WIDTH = pd.Timedelta(minutes=2.5)
        for (s, e) in violation_map.get(var, []):
            dt0, dt1 = pd.to_datetime(s), pd.to_datetime(e)
            if dt1 - dt0 < MIN_WIDTH:
                dt0 -= MIN_WIDTH; dt1 += MIN_WIDTH
            fig.add_vrect(x0=dt0, x1=dt1, fillcolor='rgba(210,30,30,0.20)',
                          line_width=0, row=row_i, col=1)

    # ── Day/night shading — only on sw/lw radiation rows ────────
    RADIATION_VARS = {'sw_in', 'sw_out', 'lw_in', 'lw_out'}
    night_shape_start = len(fig.layout.shapes)
    for row_i, var in enumerate(plot_vars, 1):
        if var not in RADIATION_VARS:
            continue
        for (ss, sr) in night_periods:
            fig.add_vrect(
                x0=ss, x1=sr,
                fillcolor='rgba(30,30,80,0.07)', line_width=0,
                layer='below', visible=False,
                row=row_i, col=1,
            )
    night_shape_count = len(fig.layout.shapes) - night_shape_start

    fig.update_layout(
        height=max(350, 260 * n),
        hovermode='x unified',
        template='plotly_white',
        title=dict(
            text=(f'Station {STATION_ID} — {", ".join(plot_vars)} — Threshold Violations<br>'
                  f'<sup>Generated {__import__("datetime").datetime.now().strftime("%Y-%m-%d %H:%M")}</sup>'),
            x=0.5, xanchor='center',
        ),
        showlegend=True,
        legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
        margin=dict(l=60, r=30, t=80, b=40),
    )
    fig.update_xaxes(rangeslider=dict(visible=False))

    js = """
(function() {
  var SHOW_DAYS = 14;
  var nightStart = """ + str(0) + """;
  var nightCount = """ + str(0) + """;

  function setNightVisible(gd, visible) {
    var update = {};
    for (var i = nightStart; i < nightStart + nightCount; i++) {
      update['shapes[' + i + '].visible'] = visible;
    }
    if (Object.keys(update).length) Plotly.relayout(gd, update);
  }

  function checkRange(gd) {
    var xa = gd.layout.xaxis;
    if (!xa || !xa.range) return;
    var days = (new Date(xa.range[1]) - new Date(xa.range[0])) / 86400000;
    setNightVisible(gd, days < SHOW_DAYS);
  }

  var gd = document.querySelector('.plotly-graph-div');
  if (!gd) return;
  gd.on('plotly_relayout', function() { checkRange(gd); });
  gd.on('plotly_doubleclick', function() { setTimeout(function() { checkRange(gd); }, 300); });
})();
""".replace('nightStart = ' + str(0), 'nightStart = ' + str(night_shape_start))    .replace('nightCount = ' + str(0), 'nightCount = ' + str(night_shape_count))

    tmp = tempfile.NamedTemporaryFile(suffix='.html', delete=False)
    html_path = tmp.name; tmp.close()
    fig.write_html(html_path, include_plotlyjs=True, full_html=True, post_script=js)
    webbrowser.open(f'file:///{html_path}')

    print(f'Chart opened in browser ({n} panel(s), {night_shape_count} night periods).')
    print('Day/night shading appears automatically when zoomed to < 14 days.')
    print('Use scroll to zoom, drag to pan, double-click to reset.')
